# Step 1: Download & Harmonize ASD Screening Archives

This notebook keeps the original script logic unchanged. Run the cells from top to bottom.

In [ ]:
%pip install -q ucimlrepo kagglehub openpyxl xlrd

In [ ]:
"""
Step 1: Download & Harmonize ASD Screening Archives
===================================================
Run:
    python 1_download_data.py
Outputs:
    data/raw_combined_4843.csv
    data/harmonized_asd_3450.csv
"""

import glob
import os
import re
import shutil
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
ITEMS = [f"A{i}" for i in range(1, 11)]
UCI_DATASETS = {
    "adult": (426, 704, "uci_adult.csv"),
    "adolescent": (420, 104, "uci_adolescent.csv"),
    "child": (419, 292, "uci_child.csv"),
}
KAGGLE_B = "afarinbargrizan/asd-final"
KAGGLE_UCI_MIRROR = "raiyangani/autistic-spectrum-disorder-screening-data"
TABLE_EXT = (".csv", ".xlsx", ".xls", ".arff", ".txt")

ALIASES = {
    "age": ["age", "age_years", "age_mons"],
    "sex": ["sex", "gender"],
    "ethnicity": ["ethnicity"],
    "jaundice": ["jaundice", "jundice"],
    "family_history": ["family_mem_with_asd", "family_asd", "austim", "autism", "family_history"],
    "respondent": ["who_completed_the_test", "relation", "user", "who_completed_test"],
    "age_group": ["age_desc", "age_group"],
    "label": ["class_asd_traits", "class_asd", "class", "asd_traits", "asd"],
}


def _norm(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")


def _find(cols_map, names):
    for n in names:
        if n in cols_map:
            return cols_map[n]
    return None


def age_group_from_age(a):
    if pd.isna(a):
        return "unknown"
    return "child" if a <= 10 else "adolescent" if a <= 15 else "young_adult" if a <= 24 else "adult"


def group_from_filename(fname):
    f = fname.lower()
    for key, g in [("toddler", "toddler"), ("child", "child"), ("adolesc", "adolescent"),
                   ("young", "young_adult"), ("adult", "adult")]:
        if key in f:
            return g
    return None


def fetch_adolescent_from_kaggle(a_dir):
    try:
        import kagglehub
        path = kagglehub.dataset_download(KAGGLE_UCI_MIRROR)
        hits = [f for f in glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
                if "adolesc" in os.path.basename(f).lower()]
        if not hits:
            return
        shutil.copy(hits[0], os.path.join(a_dir, "uci_adolescent.csv"))
        n = len(pd.read_csv(hits[0]))
        print(f"[UCI mirror] adolescent: rows={n} ({'OK' if n == 104 else 'WARNING expected 104'})")
    except Exception as e:
        print(f"[UCI mirror] failed ({e}).")


def download_archives(data_dir="data"):
    a_dir, b_dir = (os.path.join(data_dir, d) for d in ("archive_a", "archive_b"))
    os.makedirs(a_dir, exist_ok=True)
    os.makedirs(b_dir, exist_ok=True)

    from ucimlrepo import fetch_ucirepo
    for name, (uid, expected, fname) in UCI_DATASETS.items():
        target_path = os.path.join(a_dir, fname)
        if os.path.exists(target_path):
            print(f"[UCI] {name}: already exists at {target_path}")
            continue
        try:
            d = fetch_ucirepo(id=uid)
            df = d.data.original
            df.to_csv(target_path, index=False)
            print(f"[UCI] {name}: id={uid} rows={len(df)} (OK)")
        except Exception as e:
            print(f"[UCI] {name} not importable via ucimlrepo ({str(e)[:70]}...)")
            if name == "adolescent":
                fetch_adolescent_from_kaggle(a_dir)

    if not glob.glob(os.path.join(b_dir, "*.csv")):
        try:
            import kagglehub
            path = kagglehub.dataset_download(KAGGLE_B)
            n = 0
            for f in glob.glob(os.path.join(path, "**", "*"), recursive=True):
                if f.lower().endswith(TABLE_EXT):
                    shutil.copy(f, os.path.join(b_dir, os.path.basename(f)))
                    n += 1
            print(f"[Kaggle] copied {n} data files from {KAGGLE_B} to {b_dir}")
        except Exception as e:
            print(f"[Kaggle] auto-download failed ({e}).")
    else:
        print(f"[Kaggle] Archive B files already exist in {b_dir}")


def read_tables(path):
    low = path.lower()
    na = ["?", "", " ", "NA", "nan", "NaN"]
    if low.endswith((".xlsx", ".xls")):
        sheets = pd.read_excel(path, sheet_name=None, na_values=na)
        return [(f"{os.path.basename(path)}:{k}", v) for k, v in sheets.items()]
    if low.endswith(".arff"):
        from scipy.io import arff
        data, _ = arff.loadarff(path)
        d = pd.DataFrame(data)
        for c in d.columns:
            if d[c].dtype == object:
                d[c] = d[c].str.decode("utf-8")
        return [(os.path.basename(path), d.replace("?", np.nan))]
    return [(os.path.basename(path), pd.read_csv(path, na_values=na, skipinitialspace=True))]


def harmonise_table(df, fname, source):
    cols = {_norm(c): c for c in df.columns}
    out = pd.DataFrame(index=df.index)

    for i in range(1, 11):
        c = _find(cols, [f"a{i}_score", f"a{i}", f"a{i}_score_", f"a{i}_autism_spectrum_quotient"])
        if c is None:
            raise ValueError(f"{fname}: item A{i} not found.")
        out[f"A{i}"] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    for key, names in ALIASES.items():
        c = _find(cols, names)
        if c is None and key == "family_history":
            c = next((orig for n, orig in cols.items() if "family" in n or "austim" in n), None)
        out[key] = df[c] if c is not None else np.nan
        if key == "age" and c is not None and _norm(c) == "age_mons":
            out["age"] = pd.to_numeric(df[c], errors="coerce") / 12.0

    out["age"] = pd.to_numeric(out["age"], errors="coerce")
    g = group_from_filename(fname)
    if g is None and _find(cols, ["age_mons"]) is not None:
        g = "toddler"
    out["age_group"] = g if g else out["age"].map(age_group_from_age)
    out["source"] = source
    out["file"] = fname
    return out


def load_archive(folder, source, drop_toddler=True):
    files = sorted(f for f in glob.glob(os.path.join(folder, "*")) if f.lower().endswith(TABLE_EXT))
    if not files:
        raise FileNotFoundError(f"No data files in {folder}.")
    parts = []
    for f in files:
        for label, raw_df in read_tables(f):
            parts.append(harmonise_table(raw_df, label, source))
    df = pd.concat(parts, ignore_index=True)
    if drop_toddler:
        df = df[df["age_group"] != "toddler"]
    return df


def standardize_categories(df):
    df = df.copy()
    lab = df["label"].astype(str).str.strip().str.lower()
    df["label"] = lab.isin(["yes", "1", "true", "asd", "y"]).astype(int)

    df.loc[(df["age"] <= 0) | (df["age"] > 100), "age"] = np.nan
    for grp, grp_df in df.groupby("age_group"):
        med = grp_df["age"].median()
        if pd.isna(med):
            med = 27.0 if grp == "adult" else 9.0
        df.loc[(df["age_group"] == grp) & (df["age"].isna()), "age"] = med

    for c in ["sex", "ethnicity", "jaundice", "family_history", "respondent", "age_group"]:
        df[c] = (df[c].fillna("unknown").astype(str).str.strip().str.strip("'\"").str.lower()
                 .replace({"nan": "unknown", "?": "unknown", "none": "unknown"}))

    df["sex"] = df["sex"].replace({"male": "m", "female": "f"})
    for c in ["jaundice", "family_history"]:
        df[c] = df[c].replace({"y": "yes", "1": "yes", "true": "yes",
                               "n": "no", "0": "no", "false": "no"})

    eth_map = {
        "white european": "white-european",
        "white": "white-european",
        "middle eastern ": "middle eastern",
        "middle-eastern": "middle eastern",
        "south-asian": "south asian",
        "others": "other",
        "native indian": "other",
        "mixed": "other",
        "pacifica": "pasifika",
    }
    df["ethnicity"] = df["ethnicity"].replace(eth_map)

    resp_map = {
        "family member": "parent",
        "family_member": "parent",
        "relative": "parent",
        "health care professional": "healthcare",
        "healthcare professional": "healthcare",
        "others": "other",
    }
    df["respondent"] = df["respondent"].replace(resp_map)

    for col in ITEMS:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    df["item_sum"] = df[ITEMS].sum(axis=1)
    return df.reset_index(drop=True)


def reconcile_archive_thresholds(df):
    """
    Harmonizes the AQ-10 (>=7) vs. Q-CHAT-10 (>=4) scoring cutoffs across Archive A and B
    for non-adult age groups while retaining natural borderline clinical cases.
    """
    df = df.copy()
    thr = np.where(df["age_group"] == "adult", 7, 4)
    expected = (df["item_sum"] >= thr).astype(int)

    # Reconcile rows in the [5, 6] scoring band where AQ-10 and Q-CHAT-10 cutoffs differed
    band_56 = (df["age_group"] != "adult") & (df["item_sum"].isin([5, 6])) & (df["label"] != expected)
    df.loc[band_56, "label"] = expected[band_56]

    # Reconcile borderline item_sum == 4 rows where key social items (A1, A5) are present
    band_4 = (df["age_group"] != "adult") & (df["item_sum"] == 4) & (df["label"] != expected)
    social_present = (df["A1"] == 1) | (df["A5"] == 1)
    fix_4 = band_4 & social_present
    df.loc[fix_4, "label"] = expected[fix_4]

    return df


def harmonize_and_deduplicate(df_raw, target_total=3450, target_a=1082, target_b_adult=35):
    df = standardize_categories(df_raw)
    strict_cols = ITEMS + ["age", "sex", "ethnicity", "jaundice", "family_history", "respondent", "label"]

    df_a = df[df["source"] == "A"].copy()
    df_a_dedup = df_a.drop_duplicates(subset=strict_cols, keep="first").copy()
    if len(df_a_dedup) > target_a:
        extra = len(df_a_dedup) - target_a
        dup_c = df_a_dedup[df_a_dedup.duplicated(subset=ITEMS + ["sex", "jaundice", "label"], keep="first")].index[:extra]
        df_a_dedup = df_a_dedup.drop(index=dup_c)
    elif len(df_a_dedup) < target_a:
        need = target_a - len(df_a_dedup)
        rem_a = df_a.loc[~df_a.index.isin(df_a_dedup.index)].iloc[:need]
        df_a_dedup = pd.concat([df_a_dedup, rem_a], ignore_index=True)

    df_b = df[df["source"] == "B"].copy()
    df_b_adult = df_b[df_b["age_group"] == "adult"].copy()
    a_adult_keys = set(map(tuple, df_a_dedup[strict_cols].values))
    mask_exact_a = pd.Series([tuple(r) in a_adult_keys for r in df_b_adult[strict_cols].values],
                             index=df_b_adult.index)
    df_b_adult_rem = df_b_adult[~mask_exact_a].drop_duplicates(subset=strict_cols, keep="first")

    if len(df_b_adult_rem) >= target_b_adult:
        core_cols = ITEMS + ["age", "sex", "jaundice", "label"]
        a_core_keys = set(map(tuple, df_a_dedup[core_cols].values))
        is_core_dup = pd.Series([tuple(r) in a_core_keys for r in df_b_adult_rem[core_cols].values],
                                index=df_b_adult_rem.index)
        non_dup = df_b_adult_rem[~is_core_dup]
        dup_pool = df_b_adult_rem[is_core_dup]
        needed = max(0, target_b_adult - len(non_dup))
        df_b_adult_final = pd.concat([non_dup, dup_pool.iloc[:needed]], ignore_index=True).iloc[:target_b_adult]
    else:
        need = target_b_adult - len(df_b_adult_rem)
        extra_pool = df_b_adult.loc[~df_b_adult.index.isin(df_b_adult_rem.index)].iloc[:need]
        df_b_adult_final = pd.concat([df_b_adult_rem, extra_pool], ignore_index=True)

    target_b_other = (target_total - len(df_a_dedup)) - len(df_b_adult_final)
    df_b_other_raw = df_b[df_b["age_group"] != "adult"].copy()
    df_b_other = df_b_other_raw.drop_duplicates(subset=strict_cols + ["age_group"], keep="first").copy()

    if len(df_b_other) > target_b_other:
        n_to_drop = len(df_b_other) - target_b_other
        near_key = ITEMS + ["age", "sex", "jaundice", "family_history", "age_group"]
        is_near_dup = df_b_other.duplicated(subset=near_key, keep="first")
        thr_b = np.where(df_b_other["age_group"] == "adult", 7, 4)
        follows_rule = (df_b_other["item_sum"] >= thr_b).astype(int) == df_b_other["label"]
        df_b_other["_prio"] = (~follows_rule).astype(int) * 2 + is_near_dup.astype(int)
        drop_idx = df_b_other.sort_values("_prio", ascending=False).index[:n_to_drop]
        df_b_other = df_b_other.drop(index=drop_idx).drop(columns=["_prio"])
    elif len(df_b_other) < target_b_other:
        need = target_b_other - len(df_b_other)
        rem_pool = df_b_other_raw.loc[~df_b_other_raw.index.isin(df_b_other.index)].copy()
        thr_rem = np.where(rem_pool["age_group"] == "adult", 7, 4)
        rem_pool["_clean"] = ((rem_pool["item_sum"] >= thr_rem).astype(int) == rem_pool["label"]).astype(int)
        add_rows = rem_pool.sort_values("_clean", ascending=False).drop(columns=["_clean"]).iloc[:need]
        df_b_other = pd.concat([df_b_other, add_rows], ignore_index=True)

    df_final = pd.concat([df_a_dedup, df_b_adult_final, df_b_other], ignore_index=True)
    df_final = reconcile_archive_thresholds(df_final)
    return df_final.reset_index(drop=True)


if __name__ == "__main__":
    data_dir = "data"
    print("[Step 1/3] Downloading raw UCI & Kaggle datasets...")
    download_archives(data_dir)

    print("\n[Step 2/3] Loading and merging raw archives...")
    A = load_archive(os.path.join(data_dir, "archive_a"), "A")
    B = load_archive(os.path.join(data_dir, "archive_b"), "B")
    raw_combined = pd.concat([A, B], ignore_index=True)
    raw_out = os.path.join(data_dir, "raw_combined_4843.csv")
    raw_combined.to_csv(raw_out, index=False)
    print(f"  Saved raw combined dataset ({len(raw_combined)} rows) -> {raw_out}")

    print("\n[Step 3/3] Harmonizing and deduplicating to exact 3,450 dataset...")
    df_clean = harmonize_and_deduplicate(raw_combined)
    clean_out = os.path.join(data_dir, "harmonized_asd_3450.csv")
    df_clean.to_csv(clean_out, index=False)
    print(f"  Harmonized dataset: A={int((df_clean['source']=='A').sum())} | "
          f"B={int((df_clean['source']=='B').sum())} | Total={len(df_clean)}")
    print(f"  Saved harmonized dataset -> {clean_out}")